# Trainable Next-Token Transformer

This notebook trains a small causal language model on **WikiText-103 raw** using a saved BPE tokenizer. It covers data preparation, shifted next-token targets, causal multi-head self-attention, Transformer blocks, correctness checks, training, validation, generation, and checkpoint saving.

**Tensor convention:** token representations inside the model use shape `(batch, features, tokens)`. The learned token-embedding parameter is stored as `(embedding_dim, vocab_size)` to match this convention. This is an educational implementation, not a full GPT-2 reproduction or a production-optimized model.


## 0. Setup

Run this cell first. In Google Colab, select a GPU runtime when available. The notebook also works on CPU, although training may be slower. The training settings below are intentionally small so you can first verify correctness.


In [1]:
%pip install -q datasets tokenizers

import math
import time
from contextlib import nullcontext
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from tokenizers import Tokenizer

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# FP16 is the practical mixed-precision option for NVIDIA T4 GPUs.
# Set this to False to force full FP32 training for debugging/comparison.
use_fp16 = device.type == "cuda"

def autocast_context():
    """Use CUDA FP16 autocast only when enabled; otherwise use a no-op context."""
    if use_fp16:
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()

print("PyTorch version:", torch.__version__)
print("Device:", device)
print("Mixed precision:", "FP16 enabled" if use_fp16 else "disabled (FP32)")
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(device))
    print("CUDA capability:", torch.cuda.get_device_capability(device))


Note: you may need to restart the kernel to use updated packages.
PyTorch version: 2.11.0+cu128
Device: cuda
Mixed precision: FP16 enabled
GPU: Tesla T4
CUDA capability: (7, 5)


## 1. Load the tokenizer

The tokenizer was trained separately. The vocabulary size is read from the saved tokenizer file, so the model output layer always matches the tokenizer vocabulary. Do not change the tokenizer after training without retraining or carefully remapping the vocabulary parameters.


In [2]:
tokenizer_path = Path(
    "/kaggle/input/datasets/sasitumati/tokenizer/my_bpe_tokenizer(3).json"
)
if not tokenizer_path.is_file():
    raise FileNotFoundError(
        f"Tokenizer file not found: {tokenizer_path}. "
        "Update tokenizer_path to the location of your saved BPE tokenizer."
    )

tokenizer = Tokenizer.from_file(str(tokenizer_path))
vocab_size = tokenizer.get_vocab_size()
if vocab_size < 1:
    raise ValueError("The tokenizer vocabulary is empty.")

print("Tokenizer loaded:", tokenizer_path)
print("Vocabulary size:", vocab_size)


Tokenizer loaded: /kaggle/input/datasets/sasitumati/tokenizer/my_bpe_tokenizer(3).json
Vocabulary size: 32000


## 2. Load and tokenize WikiText-103

The saved BPE tokenizer converts the training and validation text into token IDs. Empty rows are skipped, and non-empty rows are separated by newline characters. Tokenization is performed in batches to limit peak memory use.


In [3]:
dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-103-raw-v1",
)

def encode_split(split, batch_rows=1000):
    if batch_rows < 1:
        raise ValueError("batch_rows must be positive.")

    token_chunks = []

    for start in range(0, len(split), batch_rows):
        rows = split[start : start + batch_rows]["text"]
        nonempty_rows = [line for line in rows if line.strip()]
        if not nonempty_rows:
            continue

        corpus_text = "\n".join(nonempty_rows)
        # Preserve a newline at the boundary between separately tokenized chunks.
        if start > 0:
            corpus_text = "\n" + corpus_text

        ids = tokenizer.encode(corpus_text).ids
        if ids:
            token_chunks.append(torch.tensor(ids, dtype=torch.int32))

    if not token_chunks:
        raise ValueError("The split produced no token IDs.")

    return torch.cat(token_chunks)

train_data = encode_split(dataset["train"])
val_data = encode_split(dataset["validation"])

# Keep compact int32 token IDs on the GPU when there is adequate free memory.
# Otherwise leave the corpus on CPU; get_batch() below transfers only sampled batches.
data_bytes = (train_data.numel() + val_data.numel()) * train_data.element_size()
keep_tokens_on_gpu = False
if device.type == "cuda":
    free_bytes, total_bytes = torch.cuda.mem_get_info(device)
    safety_margin_bytes = 1 * 1024**3
    keep_tokens_on_gpu = free_bytes >= data_bytes + safety_margin_bytes
    if keep_tokens_on_gpu:
        train_data = train_data.to(device=device, non_blocking=True)
        val_data = val_data.to(device=device, non_blocking=True)

print(f"Training tokens: {len(train_data):,}")
print(f"Validation tokens: {len(val_data):,}")
print("Token storage device:", train_data.device)
print("Token storage:", "GPU (compact int32; vectorized batch sampling)" if train_data.device.type == "cuda" else "CPU (sampled batches transferred to model device)")
print("First few training IDs:", train_data[:20].tolist())


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-103-raw-v1/test-00000-of-00001.(…): downloading bytes:           |  0.00B            

wikitext-103-raw-v1/train-00000-of-00002(…): reconstructing file:   0%|          |  0.00B /  157MB            

wikitext-103-raw-v1/train-00000-of-00002(…): downloading bytes:           |  0.00B            

...

wikitext-103-raw-v1/validation-00000-of-(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Training tokens: 117,273,053
Validation tokens: 245,674
Token storage device: cuda:0
Token storage: GPU (compact int32; vectorized batch sampling)
First few training IDs: [301, 8639, 9504, 3050, 301, 315, 199, 4720, 74, 4825, 889, 8639, 491, 529, 672, 6944, 31337, 9504, 374, 2809]


## 3. Configure the model and training run

The model predicts the next token at each input position. Input and target sequences are shifted by one token. Start with a modest training budget and increase it after the shape, gradient, and causal-mask checks pass.


In [4]:
# Architecture (preserved)
embedding_dim = 256
context_size = 128
mlp_hidden_dim = 512
num_heads = 4
num_layers = 4

# Training-speed settings:
# - Batch 16 improves utilization for this current configuration on a typical T4.
#   If you scale width/depth/context substantially, revisit this value before training.
# - Evaluate less often and with fewer batches; evaluation is forward-only work that
#   does not update the model.
batch_size = 16 if device.type == "cuda" else 4
max_iters = 50_000
learning_rate = 3e-4
weight_decay = 0.01
eval_interval = 2_000
eval_iters = 10
checkpoint_interval = 5_000
use_torch_compile = True

if embedding_dim % num_heads != 0:
    raise ValueError("embedding_dim must be divisible by num_heads.")
if context_size < 1 or batch_size < 1 or max_iters < 1:
    raise ValueError("context_size, batch_size, and max_iters must be positive.")
if eval_interval < 1 or eval_iters < 1 or checkpoint_interval < 1:
    raise ValueError("Evaluation and checkpoint intervals must be positive.")
if len(train_data) < context_size + 1 or len(val_data) < context_size + 1:
    raise ValueError(
        "Each tokenized split must contain at least context_size + 1 tokens."
    )

query_dim = key_dim = value_dim = embedding_dim // num_heads
batch_offsets = torch.arange(context_size, device=train_data.device)

print(f"Model: {embedding_dim=}, {context_size=}, {mlp_hidden_dim=}")
print(f"Attention: {num_heads} heads, {query_dim} dimensions per head")
print(f"Transformer blocks: {num_layers}")
print(f"Training: {max_iters:,} steps, batch size {batch_size}, learning rate {learning_rate}")
print(f"Tokens processed per training step: {batch_size * context_size:,}")


Model: embedding_dim=256, context_size=128, mlp_hidden_dim=512
Attention: 4 heads, 64 dimensions per head
Transformer blocks: 4
Training: 50,000 steps, batch size 16, learning rate 0.0003
Tokens processed per training step: 2,048


## 4. Create next-token training batches

For a context length of four, `x = [t0, t1, t2, t3]` and `y = [t1, t2, t3, t4]`. At each position, causal attention prevents the model from using future input tokens. Model inputs and class-index targets are returned as `torch.long` tensors.


In [5]:
def get_batch(data, batch_size=None, block_size=None, output_device=None):
    """Sample random next-token windows; defaults come from the notebook config."""
    if not isinstance(data, torch.Tensor):
        raise TypeError("data must be a torch.Tensor containing token IDs.")

    # Resolve configuration at call time (not when the function is defined).
    # This avoids NameError if cells are executed out of order and allows config changes.
    if batch_size is None:
        batch_size = globals().get("batch_size")
        if batch_size is None:
            raise RuntimeError("batch_size is undefined. Run the model/training configuration cell first.")
    if block_size is None:
        block_size = globals().get("context_size")
        if block_size is None:
            raise RuntimeError("context_size is undefined. Run the model/training configuration cell first.")
    if output_device is None:
        output_device = globals().get("device", data.device)
    output_device = torch.device(output_device)

    if not isinstance(batch_size, int) or batch_size <= 0:
        raise ValueError("batch_size must be a positive integer.")
    if not isinstance(block_size, int) or block_size <= 0:
        raise ValueError("block_size must be a positive integer.")
    if data.ndim != 1:
        raise ValueError(f"data must be a 1-D token tensor; got shape {tuple(data.shape)}.")

    # We need block_size input tokens plus one shifted target token.
    max_start = data.numel() - block_size - 1
    if max_start < 0:
        raise ValueError(
            f"Dataset has {data.numel()} tokens, but block_size={block_size} "
            "requires at least block_size + 1 tokens."
        )

    # Sample indices on the same device as the data so advanced indexing is valid.
    starts = torch.randint(0, max_start + 1, (batch_size,), device=data.device)

    cached_offsets = globals().get("batch_offsets")
    if (
        isinstance(cached_offsets, torch.Tensor)
        and cached_offsets.device == data.device
        and cached_offsets.numel() >= block_size
    ):
        offsets = cached_offsets[:block_size]
    else:
        offsets = torch.arange(block_size, device=data.device)

    positions = starts[:, None] + offsets[None, :]
    x = data[positions].to(device=output_device, dtype=torch.long)
    y = data[positions + 1].to(device=output_device, dtype=torch.long)
    return x, y


if "train_data" not in globals():
    raise RuntimeError("train_data is undefined. Run the tokenizer and dataset-loading cells first.")

x_example, y_example = get_batch(train_data)
print("Input batch shape:", tuple(x_example.shape))
print("Target batch shape:", tuple(y_example.shape))
print("Input dtype/device:", x_example.dtype, x_example.device)
print("Target dtype/device:", y_example.dtype, y_example.device)
print("First input sequence:", x_example[0, :10].tolist())
print("Shifted targets:     ", y_example[0, :10].tolist())
assert x_example.shape == y_example.shape, "Input and target batches must have the same shape."
assert x_example.dtype == torch.long and y_example.dtype == torch.long
assert torch.equal(x_example[:, 1:], y_example[:, :-1]), "Input/target shift check failed."
print("Batch sampling and next-token alignment checks passed.")

Input batch shape: (16, 128)
Target batch shape: (16, 128)
Input dtype/device: torch.int64 cuda:0
Target dtype/device: torch.int64 cuda:0
First input sequence: [737, 331, 83, 4448, 1491, 83, 403, 261, 645, 394]
Shifted targets:      [331, 83, 4448, 1491, 83, 403, 261, 645, 394, 5046]
Batch sampling and next-token alignment checks passed.


## 5. Define the configurable Transformer

Each block uses Pre-LayerNorm, causal multi-head self-attention with a residual connection, and a feed-forward MLP with ReLU and a second residual connection. The forward pass returns raw logits shaped `(batch, vocab_size, tokens)` for next-token cross-entropy training.


In [7]:



# ============================================================
# 1. MULTI-HEAD CAUSAL SELF-ATTENTION
# ============================================================

class MultiHeadCausalAttention(nn.Module):
    def __init__(self, embedding_dim, num_heads):
        super().__init__()

        if num_heads < 1:
            raise ValueError("num_heads must be at least 1.")
        if embedding_dim % num_heads != 0:
            raise ValueError("embedding_dim must be divisible by num_heads.")

        self.embedding_dim = embedding_dim
        self.num_heads = num_heads
        self.head_dim = embedding_dim // num_heads
        self.query_dim = self.key_dim = self.value_dim = self.head_dim

        # One matrix multiplication computes Q, K and V together.
        # bias=False preserves the original no-bias Q/K/V projections.
        self.qkv = nn.Linear(embedding_dim, 3 * embedding_dim, bias=False)
        nn.init.normal_(self.qkv.weight, mean=0.0, std=0.02)

        # Output projection is unchanged.
        self.W_O = nn.Parameter(torch.randn(embedding_dim, embedding_dim) * 0.02)

    def forward(self, x):
        """Input/output: (batch, embedding_dim, tokens)."""
        batch, _, n = x.shape

        # Linear expects (batch, tokens, embedding_dim).
        x_tokens = x.transpose(1, 2)
        qkv = self.qkv(x_tokens)
        qkv = qkv.view(batch, n, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0)

        if hasattr(F, "scaled_dot_product_attention"):
            head_outputs = F.scaled_dot_product_attention(
                q, k, v,
                attn_mask=None,
                dropout_p=0.0,
                is_causal=True,
            )
        else:
            # Compatibility fallback for older PyTorch releases.
            scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
            future_mask = torch.triu(
                torch.ones(n, n, dtype=torch.bool, device=x.device), diagonal=1
            )
            scores = scores.masked_fill(future_mask[None, None, :, :], float("-inf"))
            attention_probs = torch.softmax(scores, dim=-1)
            head_outputs = torch.matmul(attention_probs, v)

        combined = head_outputs.transpose(1, 2).reshape(batch, n, self.embedding_dim)
        combined = combined.transpose(1, 2)
        return self.W_O @ combined


# ============================================================
# 2. ONE TRANSFORMER BLOCK
# ============================================================

class TransformerBlock(nn.Module):
    def __init__(
        self,
        embedding_dim,
        num_heads,
        mlp_hidden_dim,
    ):
        super().__init__()

        self.attention_norm = nn.LayerNorm(embedding_dim)
        self.attention = MultiHeadCausalAttention(
            embedding_dim=embedding_dim,
            num_heads=num_heads,
        )

        self.mlp_norm = nn.LayerNorm(embedding_dim)

        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim, mlp_hidden_dim),
            nn.ReLU(),
            nn.Linear(mlp_hidden_dim, embedding_dim),
        )

    def forward(self, x):
        """
        x: (batch, embedding_dim, tokens)
        """

        # Pre-LayerNorm -> attention -> residual connection
        normalized = self.attention_norm(x.transpose(1, 2))
        normalized = normalized.transpose(1, 2)

        x = x + self.attention(normalized)

        # Pre-LayerNorm -> MLP -> residual connection
        normalized = self.mlp_norm(x.transpose(1, 2))
        mlp_output = self.mlp(normalized).transpose(1, 2)

        x = x + mlp_output

        return x


# ============================================================
# 3. CONFIGURABLE MULTI-HEAD, MULTI-LAYER TRANSFORMER
# ============================================================

class TinyCausalTransformer(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim,
        context_size,
        mlp_hidden_dim,
        num_heads,
        num_layers,
    ):
        super().__init__()

        if embedding_dim < 1:
            raise ValueError("embedding_dim must be positive.")
        if context_size < 1:
            raise ValueError("context_size must be positive.")
        if mlp_hidden_dim < 1:
            raise ValueError("mlp_hidden_dim must be positive.")
        if num_layers < 1:
            raise ValueError("num_layers must be at least 1.")
        if num_heads < 1:
            raise ValueError("num_heads must be at least 1.")
        if embedding_dim % num_heads != 0:
            raise ValueError(
                "embedding_dim must be divisible by num_heads."
            )

        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.context_size = context_size
        self.mlp_hidden_dim = mlp_hidden_dim
        self.num_heads = num_heads
        self.num_layers = num_layers

        # Automatically derived per-head dimensions.
        self.query_dim = embedding_dim // num_heads
        self.key_dim = embedding_dim // num_heads
        self.value_dim = embedding_dim // num_heads
        self.head_dim = self.query_dim

        # Learned embeddings: same convention as your original model.
        self.token_embedding = nn.Parameter(
            torch.randn(embedding_dim, vocab_size) * 0.02
        )
        self.position_embedding = nn.Parameter(
            torch.randn(embedding_dim, context_size) * 0.02
        )

        # Each block has its own attention and MLP parameters.
        self.blocks = nn.ModuleList([
            TransformerBlock(
                embedding_dim=embedding_dim,
                num_heads=num_heads,
                mlp_hidden_dim=mlp_hidden_dim,
            )
            for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(embedding_dim)

        # Vocabulary projection.
        self.W_vocab = nn.Parameter(
            torch.randn(vocab_size, embedding_dim) * 0.02
        )
        self.b_vocab = nn.Parameter(torch.zeros(vocab_size, 1))

    def forward(self, input_ids):
        """
        input_ids: (batch, tokens)
        logits:    (batch, vocab_size, tokens)
        """
        if input_ids.ndim != 2:
            raise ValueError(
                "input_ids must have shape (batch, tokens)."
            )

        batch, n = input_ids.shape

        if n < 1 or n > self.context_size:
            raise ValueError(
                f"Sequence length must be between 1 and "
                f"{self.context_size}."
            )

        # Token and positional embeddings.
        token_vectors = self.token_embedding[:, input_ids].permute(1, 0, 2)
        position_vectors = self.position_embedding[:, :n].unsqueeze(0)
        x = token_vectors + position_vectors

        # Pass through the selected number of Transformer blocks.
        for block in self.blocks:
            x = block(x)

        # Final normalization.
        x = self.final_norm(x.transpose(1, 2)).transpose(1, 2)

        # Next-token vocabulary logits.
        logits = self.W_vocab @ x + self.b_vocab

        return logits

    @torch.no_grad()
    def generate(
        self,
        input_ids,
        max_new_tokens=40,
        temperature=1.0,
        top_k=None,
    ):
        """Generate tokens autoregressively."""
        if temperature <= 0:
            raise ValueError("temperature must be greater than zero.")
        if max_new_tokens < 0:
            raise ValueError("max_new_tokens must be non-negative.")
        if top_k is not None and int(top_k) <= 0:
            raise ValueError("top_k must be positive or None.")
        if input_ids.ndim != 2 or input_ids.shape[1] == 0:
            raise ValueError(
                "input_ids must have shape (batch, non-empty tokens)."
            )

        was_training = self.training
        self.eval()
        try:
            for _ in range(max_new_tokens):
                idx = input_ids[:, -self.context_size:]

                # Autocast accelerates inference on CUDA; sample in FP32 for stability.
                with autocast_context():
                    logits = self(idx)[:, :, -1]
                logits = logits.float() / temperature

                if top_k is not None:
                    k = min(int(top_k), logits.shape[-1])
                    threshold = torch.topk(
                        logits, k, dim=-1
                    ).values[:, -1, None]
                    logits = logits.masked_fill(
                        logits < threshold, float("-inf")
                    )

                probs = torch.softmax(logits, dim=-1)
                next_id = torch.multinomial(probs, num_samples=1)
                input_ids = torch.cat([input_ids, next_id], dim=1)
        finally:
            self.train(was_training)

        return input_ids


In [8]:
# Instantiate the model from the configuration above.
model_config = {
    "vocab_size": vocab_size,
    "embedding_dim": embedding_dim,
    "context_size": context_size,
    "mlp_hidden_dim": mlp_hidden_dim,
    "num_heads": num_heads,
    "num_layers": num_layers,
}

base_model = TinyCausalTransformer(**model_config).to(device)
model = base_model

print("Attention heads:", base_model.num_heads)
print("Transformer blocks:", base_model.num_layers)
print("Dimension per head:", base_model.head_dim)
print("Trainable parameters:", f"{sum(p.numel() for p in base_model.parameters() if p.requires_grad):,}")
print("Model device:", next(base_model.parameters()).device)


Attention heads: 4
Transformer blocks: 4
Dimension per head: 64
Trainable parameters: 18,553,600
Model device: cuda:0


## Attention backend check: SDPA and FlashAttention

This lightweight check runs before training. It verifies that the regular SDPA call works and separately tests whether PyTorch can run the FlashAttention backend for the current device, dtype, head count, context length, and head dimension. It does **not** force FlashAttention for training; the model continues calling SDPA normally, so PyTorch can choose an eligible backend.

In [9]:
def check_attention_backends():
    """Smoke-test automatic SDPA and explicitly test FlashAttention if CUDA is available."""
    if not hasattr(F, "scaled_dot_product_attention"):
        print("SDPA is not available in this PyTorch version; model will use its explicit attention fallback.")
        return

    test_batch = min(max(int(batch_size), 1), 2)
    head_dim = embedding_dim // num_heads
    test_dtype = torch.float16 if (device.type == "cuda" and use_fp16) else torch.float32
    shape = (test_batch, num_heads, context_size, head_dim)

    # Test regular SDPA (the same API the model uses); PyTorch chooses the backend.
    try:
        q = torch.randn(shape, device=device, dtype=test_dtype)
        k = torch.randn(shape, device=device, dtype=test_dtype)
        v = torch.randn(shape, device=device, dtype=test_dtype)
        with torch.no_grad():
            auto_out = F.scaled_dot_product_attention(
                q, k, v, dropout_p=0.0, is_causal=True
            )
        if device.type == "cuda":
            torch.cuda.synchronize(device)
        assert auto_out.shape == q.shape
        print(f"Automatic SDPA check: PASS (shape={shape}, dtype={test_dtype}, device={device}).")
        print("Note: this confirms SDPA works; it does not identify which backend SDPA selected.")
    except Exception as exc:
        print(f"Automatic SDPA check: FAILED ({type(exc).__name__}: {exc}).")
        return

    if device.type != "cuda":
        print("FlashAttention check: SKIPPED because CUDA is not available. CPU SDPA passed.")
        return

    # Prefer the modern API. Older PyTorch 2.x versions expose the same control
    # through torch.backends.cuda.sdp_kernel instead.
    try:
        from torch.nn.attention import sdpa_kernel, SDPBackend
        flash_context = lambda: sdpa_kernel(SDPBackend.FLASH_ATTENTION)
        api_name = "torch.nn.attention.sdpa_kernel"
    except (ImportError, AttributeError):
        legacy_sdp_kernel = getattr(torch.backends.cuda, "sdp_kernel", None)
        if legacy_sdp_kernel is None:
            print("FlashAttention check: UNAVAILABLE; this PyTorch version exposes no backend-selection API.")
            return
        flash_context = lambda: legacy_sdp_kernel(
            enable_flash=True, enable_math=False, enable_mem_efficient=False
        )
        api_name = "torch.backends.cuda.sdp_kernel (legacy API)"

    # Restrict dispatch to FlashAttention for this call. If unsupported, the call
    # should fail rather than silently using math/memory-efficient attention.
    import warnings
    try:
        with warnings.catch_warnings(record=True) as caught_warnings:
            warnings.simplefilter("always")
            with flash_context():
                with torch.no_grad():
                    flash_out = F.scaled_dot_product_attention(
                        q, k, v, dropout_p=0.0, is_causal=True
                    )
            torch.cuda.synchronize(device)
        assert flash_out.shape == q.shape
        print(f"FlashAttention backend check: PASS via {api_name} for this device, dtype, and tensor shape.")
        if caught_warnings:
            print("FlashAttention warning:", str(caught_warnings[0].message)[:500])
        print("Important: normal SDPA may choose a different eligible backend; this test only confirms FlashAttention can run when explicitly requested.")
    except Exception as exc:
        print("FlashAttention backend check: NOT AVAILABLE for this device/dtype/shape.")
        print(f"Reason: {type(exc).__name__}: {str(exc)[:700]}")
        print("The model can still use another SDPA backend; no separate FlashAttention package is required for this check.")


check_attention_backends()

Automatic SDPA check: PASS (shape=(2, 4, 128, 64), dtype=torch.float16, device=cuda).
Note: this confirms SDPA works; it does not identify which backend SDPA selected.
FlashAttention backend check: NOT AVAILABLE for this device/dtype/shape.
Reason: RuntimeError: No available kernel. Aborting execution.
The model can still use another SDPA backend; no separate FlashAttention package is required for this check.


### Efficient attention and fused projections

The attention layer computes Q, K and V with one bias-free linear projection, reducing three separate projection operations to one. Attention itself uses PyTorch SDPA, which selects a supported backend based on hardware, dtype and tensor layout; this does not guarantee that the FlashAttention kernel is used. The backend-check cell below tests whether FlashAttention can run with this configuration, without forcing it during training. A legacy-checkpoint migration below can combine the old separate Q/K/V weights into the fused projection.


## 6. Check shapes, gradients, and causal masking

This test verifies the output shape, confirms that all trainable parameters receive gradients, and checks that changing a token cannot affect logits at earlier positions.


In [10]:
# Smoke test: output shape and gradient flow
model.train()
logits = model(x_example)
assert logits.shape == (batch_size, vocab_size, context_size), (
    f"Unexpected logits shape: {tuple(logits.shape)}"
)

# Cross-entropy accepts (batch, classes, tokens) directly. Avoid permute+reshape,
# which would materialize a large temporary copy of the vocabulary logits.
smoke_loss = F.cross_entropy(logits, y_example)
smoke_loss.backward()
missing_gradients = [
    name for name, parameter in model.named_parameters()
    if parameter.requires_grad and parameter.grad is None
]
assert not missing_gradients, f"Parameters without gradients: {missing_gradients}"
print("Forward-pass shape:", tuple(logits.shape))
print("Smoke-test loss:", round(smoke_loss.item(), 4))
print("Gradient check: passed")
model.zero_grad(set_to_none=True)

# Causality test: modifying token t must not affect earlier logits.
model.eval()
x_original = x_example[0:1].clone()
x_changed = x_original.clone()
t = max(1, context_size // 2)
x_changed[0, t] = (x_changed[0, t] + 1) % vocab_size

with torch.no_grad():
    logits_original = model(x_original)
    logits_changed = model(x_changed)

causal_ok = torch.allclose(
    logits_original[:, :, :t],
    logits_changed[:, :, :t],
    atol=1e-6,
    rtol=1e-5,
)
print(f"Causal masking check (changed position {t}):", causal_ok)
assert causal_ok, (
    "Changing a later token altered earlier-position logits; "
    "inspect the attention mask."
)
model.train()


Forward-pass shape: (16, 32000, 128)
Smoke-test loss: 10.4237
Gradient check: passed
Causal masking check (changed position 64): True


TinyCausalTransformer(
  (blocks): ModuleList(
    (0-3): 4 x TransformerBlock(
      (attention_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (attention): MultiHeadCausalAttention(
        (qkv): Linear(in_features=256, out_features=768, bias=False)
      )
      (mlp_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (mlp): Sequential(
        (0): Linear(in_features=256, out_features=512, bias=True)
        (1): ReLU()
        (2): Linear(in_features=512, out_features=256, bias=True)
      )
    )
  )
  (final_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
)

## 7. Create the optimizer and verify parameter registration

`AdamW` updates all registered trainable parameters. The ID-set check confirms that no trainable parameter is accidentally omitted from the optimizer.


In [11]:
trainable_parameters = [p for p in base_model.parameters() if p.requires_grad]
optimizer_backend = "standard"

if device.type == "cuda":
    try:
        optimizer = torch.optim.AdamW(
            trainable_parameters,
            lr=learning_rate,
            weight_decay=weight_decay,
            fused=True,
        )
        optimizer_backend = "fused"
    except (TypeError, RuntimeError, ValueError, NotImplementedError):
        try:
            optimizer = torch.optim.AdamW(
                trainable_parameters,
                lr=learning_rate,
                weight_decay=weight_decay,
                foreach=True,
            )
            optimizer_backend = "foreach"
        except (TypeError, RuntimeError, ValueError, NotImplementedError):
            optimizer = torch.optim.AdamW(
                trainable_parameters,
                lr=learning_rate,
                weight_decay=weight_decay,
            )
else:
    optimizer = torch.optim.AdamW(
        trainable_parameters,
        lr=learning_rate,
        weight_decay=weight_decay,
    )

# GradScaler protects FP16 gradients from underflow; disabled scaler is a no-op.
try:
    grad_scaler = torch.amp.GradScaler("cuda", enabled=use_fp16)
except (AttributeError, TypeError):
    grad_scaler = torch.cuda.amp.GradScaler(enabled=use_fp16)

model_parameter_ids = {id(p) for p in trainable_parameters}
optimizer_parameter_ids = {
    id(parameter)
    for group in optimizer.param_groups
    for parameter in group["params"]
}
assert model_parameter_ids == optimizer_parameter_ids, (
    "Optimizer parameter groups do not match model parameters."
)
print("Optimizer:", type(optimizer).__name__, f"({optimizer_backend})")
print("GradScaler enabled:", grad_scaler.is_enabled())
print(
    "Parameters registered with optimizer:",
    f"{sum(parameter.numel() for group in optimizer.param_groups for parameter in group['params']):,}",
)


Optimizer: AdamW (fused)
GradScaler enabled: True
Parameters registered with optimizer: 18,553,600


## 8. Estimate training and validation loss

Cross-entropy is computed over all target positions. Validation batches are used only for evaluation and do not contribute gradients. The function restores the model's previous train/eval mode even if evaluation raises an error.


In [12]:
@torch.no_grad()
def estimate_loss():
    was_training = model.training
    model.eval()
    results = {}

    try:
        for split_name, split_data in (("train", train_data), ("validation", val_data)):
            losses = []
            for _ in range(eval_iters):
                xb, yb = get_batch(split_data)
                with autocast_context():
                    logits = model(xb)
                    # logits already use (batch, vocab_size, tokens); no reshape copy.
                    loss = F.cross_entropy(logits, yb)
                losses.append(loss.float().item())
            results[split_name] = sum(losses) / len(losses)
    finally:
        model.train(was_training)

    return results

initial_losses = estimate_loss()
print("Initial average loss:", initial_losses)


Initial average loss: {'train': 10.435174942016602, 'validation': 10.439645290374756}


## 9. Train the model

Each step performs a forward pass, cross-entropy calculation, backpropagation, gradient clipping, and an `AdamW` update. Periodic loss values are averages over independently sampled batches and therefore have some sampling noise.


### Training-speed changes in this version

- Fused, bias-free QKV projection: one linear operation computes queries, keys and values.
- Optional `torch.compile` on CUDA, enabled by default when available; unsupported graph fragments are configured to fall back to eager execution.
- FP16 autocast with gradient scaling, SDPA causal attention, and fused/foreach AdamW where supported.
- Vectorized token-window sampling and GPU-resident token storage when memory permits.
- Cross-entropy consumes `(batch, vocab_size, tokens)` logits directly to avoid a large permutation/reshape copy.
- CUDA batch size is set to 16 for this current configuration; lower it if you later scale the model/context and run out of memory.
- Evaluation is reduced to every 2,000 steps with 10 batches per split to reduce forward-only overhead. Final evaluation is still performed.
- Checkpoints save the uncompiled model state. Older checkpoints with separate Q/K/V weights can be migrated; their optimizer moments are reset during migration.


In [13]:
# Set True only when continuing from a compatible checkpoint.
resume_from_checkpoint = False
checkpoint_name = "checkpoint_dim256_heads4_layers4_step20000.pt"
resume_checkpoint_path = Path(checkpoint_name)
legacy_qkv_migrated = False

if resume_from_checkpoint:
    if not resume_checkpoint_path.is_file():
        raise FileNotFoundError(
            f"Checkpoint not found: {resume_checkpoint_path}. "
            "Choose a checkpoint created by this model architecture."
        )

    checkpoint = torch.load(
        resume_checkpoint_path,
        map_location=device,
        weights_only=False,
    )
    saved_config = checkpoint.get("config")
    if saved_config is not None and saved_config != model_config:
        raise ValueError(
            "Checkpoint architecture does not match the current configuration. "
            f"Saved config: {saved_config}; current config: {model_config}"
        )
    if "model_state_dict" not in checkpoint or "optimizer_state_dict" not in checkpoint:
        raise ValueError("Checkpoint must contain model and optimizer state dictionaries.")

    # Convert prior checkpoints that stored W_Q, W_K and W_V separately.
    # The concatenated tensor has the same Q/K/V projection values and ordering.
    model_state = dict(checkpoint["model_state_dict"])
    for layer_index in range(base_model.num_layers):
        prefix = f"blocks.{layer_index}.attention."
        q_key, k_key, v_key = prefix + "W_Q", prefix + "W_K", prefix + "W_V"
        fused_key = prefix + "qkv.weight"
        if q_key in model_state and k_key in model_state and v_key in model_state:
            q_weight = model_state.pop(q_key).reshape(embedding_dim, embedding_dim)
            k_weight = model_state.pop(k_key).reshape(embedding_dim, embedding_dim)
            v_weight = model_state.pop(v_key).reshape(embedding_dim, embedding_dim)
            model_state[fused_key] = torch.cat([q_weight, k_weight, v_weight], dim=0)
            legacy_qkv_migrated = True

    try:
        base_model.load_state_dict(model_state)
        if not legacy_qkv_migrated:
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
            if "grad_scaler_state_dict" in checkpoint:
                grad_scaler.load_state_dict(checkpoint["grad_scaler_state_dict"])
        else:
            print(
                "Migrated legacy Q/K/V weights to fused QKV. The optimizer state was "
                "reset because its parameter layout changed; model weights were retained."
            )
    except (RuntimeError, KeyError, ValueError) as exc:
        raise ValueError(
            "Could not resume from this checkpoint. Ensure it matches this model and optimizer."
        ) from exc

    start_step = int(checkpoint["step"])
    training_history = checkpoint.get("training_history", [])
    print(f"Resuming from step {start_step}: {resume_checkpoint_path}")
else:
    start_step = 0
    training_history = []
    print("Starting from scratch at step 0.")

# Apply the current learning-rate setting, including when resuming.
for group in optimizer.param_groups:
    group["lr"] = learning_rate
print("Learning rate:", optimizer.param_groups[0]["lr"])

# Compile only after any checkpoint has been loaded. Use Dynamo's eager fallback
# for unsupported graph fragments so compilation issues do not stop training.
model = base_model
if use_torch_compile and device.type == "cuda" and hasattr(torch, "compile"):
    try:
        try:
            import torch._dynamo
            torch._dynamo.config.suppress_errors = True
        except Exception:
            pass
        model = torch.compile(base_model, mode="default")
        print("torch.compile enabled (mode='default'; compile startup is included in total run time).")
    except Exception as exc:
        model = base_model
        print(f"torch.compile could not be enabled; using eager execution ({type(exc).__name__}: {exc}).")
else:
    print("torch.compile unavailable, disabled, or not using CUDA; using eager execution.")


Starting from scratch at step 0.
Learning rate: 0.0003
torch.compile enabled (mode='default'; compile startup is included in total run time).


In [14]:
# Train and save checkpoints
# Timing is collected inside the normal training run; no extra optimizer updates are added.
# The first 10 steps are warm-up and the next 50 are timed.
benchmark_warmup_steps = min(10, max(0, max_iters - start_step))
benchmark_steps = min(50, max(0, max_iters - start_step - benchmark_warmup_steps))
benchmark_start_step = start_step + benchmark_warmup_steps
benchmark_start_time = None
benchmark_reported = False

start_time = time.time()
model.train()

for step in range(start_step, max_iters):
    if step % eval_interval == 0:
        losses = estimate_loss()
        training_history.append({"step": step, **losses})
        print(
            f"step {step:5d} | "
            f"train loss {losses['train']:.4f} | "
            f"validation loss {losses['validation']:.4f}",
            flush=True,
        )

    model.train()

    # Synchronize only at benchmark-window boundaries so timing reflects a run of steps,
    # not the overhead of synchronizing after every GPU operation.
    if benchmark_steps > 0 and step == benchmark_start_step:
        if device.type == "cuda":
            torch.cuda.synchronize(device)
            torch.cuda.reset_peak_memory_stats(device)
        benchmark_start_time = time.perf_counter()

    xb, yb = get_batch(train_data)
    with autocast_context():
        logits = model(xb)
        # Keep the native (batch, vocab_size, tokens) layout to avoid a large copy.
        loss = F.cross_entropy(logits, yb)

    optimizer.zero_grad(set_to_none=True)
    grad_scaler.scale(loss).backward()
    # Unscale before clipping so max_norm=1.0 acts on real gradients.
    grad_scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(base_model.parameters(), max_norm=1.0)
    grad_scaler.step(optimizer)
    grad_scaler.update()

    if (
        benchmark_steps > 0
        and not benchmark_reported
        and step == benchmark_start_step + benchmark_steps - 1
    ):
        if device.type == "cuda":
            torch.cuda.synchronize(device)
        benchmark_elapsed = time.perf_counter() - benchmark_start_time
        measured_tokens = benchmark_steps * batch_size * context_size
        print("\n--- Training efficiency snapshot ---")
        print(f"Device: {device}")
        if device.type == "cuda":
            print(f"GPU: {torch.cuda.get_device_name(device)}")
            peak_allocated_gib = torch.cuda.max_memory_allocated(device) / (1024 ** 3)
            peak_reserved_gib = torch.cuda.max_memory_reserved(device) / (1024 ** 3)
            print(f"Peak allocated GPU memory: {peak_allocated_gib:.2f} GiB")
            print(f"Peak reserved GPU memory:  {peak_reserved_gib:.2f} GiB")
        print(f"Warm-up steps excluded from timing: {benchmark_warmup_steps}")
        print(f"Timed training steps: {benchmark_steps}")
        print(f"Average step time: {benchmark_elapsed / benchmark_steps:.4f} s")
        print(f"Training throughput: {measured_tokens / benchmark_elapsed:,.0f} tokens/s")
        print("Note: throughput includes batch creation, forward/backward, and optimizer update.")
        print("------------------------------------\n")
        benchmark_reported = True

    if (step + 1) % checkpoint_interval == 0:
        checkpoint_path = (
            f"checkpoint_dim{embedding_dim}_heads{num_heads}"
            f"_layers{num_layers}_step{step + 1}.pt"
        )
        torch.save(
            {
                "step": step + 1,
                "model_state_dict": base_model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "grad_scaler_state_dict": grad_scaler.state_dict(),
                "training_history": training_history,
                "config": model_config,
            },
            checkpoint_path,
        )
        print(f"Saved checkpoint: {checkpoint_path}", flush=True)

final_losses = estimate_loss()
training_history.append({"step": max_iters, **final_losses})
elapsed = time.time() - start_time

print(f"\nFinal evaluation at {max_iters:,} steps")
print(f"Training loss:   {final_losses['train']:.4f}")
print(f"Validation loss: {final_losses['validation']:.4f}")

final_checkpoint_path = (
    f"checkpoint_dim{embedding_dim}_heads{num_heads}"
    f"_layers{num_layers}_step{max_iters}.pt"
)
torch.save(
    {
        "step": max_iters,
        "model_state_dict": base_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "grad_scaler_state_dict": grad_scaler.state_dict(),
        "training_history": training_history,
        "final_losses": final_losses,
        "config": model_config,
        "elapsed_seconds": elapsed,
    },
    final_checkpoint_path,
)
print(f"Final checkpoint saved: {final_checkpoint_path}")
print(f"Elapsed time: {elapsed / 60:.2f} minutes")


W1021 09:41:29.064000 48 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


step     0 | train loss 10.4368 | validation loss 10.4373

--- Training efficiency snapshot ---
Device: cuda
GPU: Tesla T4
Peak allocated GPU memory: 1.75 GiB
Peak reserved GPU memory:  1.89 GiB
Warm-up steps excluded from timing: 10
Timed training steps: 50
Average step time: 0.0694 s
Training throughput: 29,501 tokens/s
Note: throughput includes batch creation, forward/backward, and optimizer update.
------------------------------------

step  2000 | train loss 5.5504 | validation loss 5.5344
step  4000 | train loss 5.1822 | validation loss 5.0829
Saved checkpoint: checkpoint_dim256_heads4_layers4_step5000.pt
step  6000 | train loss 4.9337 | validation loss 4.8539
step  8000 | train loss 4.8071 | validation loss 4.7613
Saved checkpoint: checkpoint_dim256_heads4_layers4_step10000.pt
step 10000 | train loss 4.7314 | validation loss 4.6543
step 12000 | train loss 4.6374 | validation loss 4.4668
step 14000 | train loss 4.4883 | validation loss 4.4407
Saved checkpoint: checkpoint_dim256_h

## 10. Evaluate final loss and perplexity

Perplexity is `exp(validation cross-entropy loss)`. It depends strongly on the tokenizer, data, and prediction objective, so compare it only between experiments with compatible setups.


In [15]:
val_loss = final_losses["validation"]
val_perplexity = math.exp(val_loss) if val_loss < 20 else float("inf")

print("Final losses:", final_losses)
print("Validation perplexity:", round(val_perplexity, 3))
print("Loss history:")
for item in training_history:
    print(item)


Final losses: {'train': 4.126938033103943, 'validation': 4.007500314712525}
Validation perplexity: 55.009
Loss history:
{'step': 0, 'train': 10.436807441711426, 'validation': 10.43731107711792}
{'step': 2000, 'train': 5.5504169940948485, 'validation': 5.534417343139649}
{'step': 4000, 'train': 5.182154417037964, 'validation': 5.082901334762573}
{'step': 6000, 'train': 4.933740425109863, 'validation': 4.853925037384033}
{'step': 8000, 'train': 4.807069587707519, 'validation': 4.761279439926147}
{'step': 10000, 'train': 4.7313549518585205, 'validation': 4.654298448562622}
{'step': 12000, 'train': 4.637449121475219, 'validation': 4.466826772689819}
{'step': 14000, 'train': 4.488255262374878, 'validation': 4.4406767845153805}
{'step': 16000, 'train': 4.383621549606323, 'validation': 4.350596714019775}
{'step': 18000, 'train': 4.398510932922363, 'validation': 4.461250495910645}
{'step': 20000, 'train': 4.338759040832519, 'validation': 4.335221576690674}
{'step': 22000, 'train': 4.2666464328

## 11. Generate sample continuations

The generator repeatedly predicts and samples the next token. Temperature changes the distribution's sharpness; top-k restricts sampling to the highest-scoring tokens. These settings do not update the model weights. Because this is a small model with a limited training budget, generated text may be repetitive or incoherent.


In [16]:
# Test the trained Transformer on several prompts
prompts = [
    "India got independence in",
    "During the First World War, soldiers lived in muddy trenches for weeks, facing cold weather and constant danger. On Christmas Eve, something unexpected happened when soldiers on opposite sides began to",
    "A small town built a new railway station to reduce traffic congestion. At first, people were excited about the faster transport, but after several months the roads became even more crowded because",
    "An engineer developed a robot that could deliver packages without human assistance. During its first week in a busy city, the robot repeatedly stopped at the same street corner, where its sensors detected",
    "A young researcher discovered that an old computer in an abandoned laboratory was still receiving messages every night. When she examined the messages, she noticed that each one contained information about events that had not yet",
]

for i, prompt in enumerate(prompts, start=1):
    input_ids_list = tokenizer.encode(prompt).ids
    if not input_ids_list:
        print(f"Skipping prompt {i}: tokenizer returned no tokens.")
        continue

    input_ids = torch.tensor(
        [input_ids_list], dtype=torch.long, device=device
    )
    generated_ids = base_model.generate(
        input_ids,
        max_new_tokens=50,
        temperature=0.8,
        top_k=20,
    )
    generated_text = tokenizer.decode(generated_ids[0].tolist())

    print(f"\n{'=' * 70}")
    print(f"Prompt {i}: {prompt}")
    print("-" * 70)
    print(generated_text)



Prompt 1: India got independence in
----------------------------------------------------------------------
India got independence in March 1979 , and the song was recorded on the UK Singles Chart , making it number @-@ one on the UK Singles Chart . It peaked at number 10 on the Canadian Hot 100 . It was also certified platinum by the European Hot 100 . The song also

Prompt 2: During the First World War, soldiers lived in muddy trenches for weeks, facing cold weather and constant danger. On Christmas Eve, something unexpected happened when soldiers on opposite sides began to
----------------------------------------------------------------------
During the First World War, soldiers lived in muddy trenches for weeks, facing cold weather and constant danger. On Christmas Eve, something unexpected happened when soldiers on opposite sides began to evacuate at home . 

 = = = Post @-@ war = = = 

 The First Coalition ( NORS ) launched a series of attacks against the advancing German forces 

In [18]:
import time
import torch
import json
import os

# ---- EDIT these if your notebook uses different variable names ----
model = model
optimizer = optimizer

# ---- Training configuration ----
results = {
    "training_steps": 50_000,
    "training_time_minutes": 60,
    "parameters_total": sum(p.numel() for p in model.parameters()),
    "parameters_trainable": sum(
        p.numel() for p in model.parameters() if p.requires_grad
    ),
    "device": str(next(model.parameters()).device),
}

# ---- Model configuration: capture if available ----
if hasattr(model, "config"):
    results["model_config"] = str(model.config)

# ---- Optimizer configuration ----
if optimizer is not None:
    results["optimizer"] = optimizer.__class__.__name__
    results["learning_rates"] = [
        group["lr"] for group in optimizer.param_groups
    ]

# ---- GPU metrics ----
if torch.cuda.is_available():
    results["gpu_name"] = torch.cuda.get_device_name(0)
    results["peak_gpu_memory_allocated_GB"] = round(
        torch.cuda.max_memory_allocated(0) / 1024**3, 3
    )
    results["peak_gpu_memory_reserved_GB"] = round(
        torch.cuda.max_memory_reserved(0) / 1024**3, 3
    )

# ---- Training results: fill in from your logs ----
results.update({
    "final_train_loss": None,  # enter your final logged value
    "final_val_loss": None,    # enter your final logged value
    "dataset": "WikiText-103", # correct this if needed
    "sequence_length": 128,    # correct if needed
    "batch_size": None,        # enter your actual batch size
    "learning_rate_schedule": None,
    "precision": None,         # e.g. float32, float16, bfloat16
})

print(json.dumps(results, indent=2))

with open("baseline_50k.json", "w") as f:
    json.dump(results, f, indent=2)

print("\nSaved baseline_50k.json")

{
  "training_steps": 50000,
  "training_time_minutes": 60,
  "parameters_total": 18553600,
  "parameters_trainable": 18553600,
  "device": "cuda:0",
  "optimizer": "AdamW",
  "learning_rates": [
    0.0003
  ],
  "gpu_name": "Tesla T4",
  "peak_gpu_memory_allocated_GB": 1.746,
  "peak_gpu_memory_reserved_GB": 1.887,
  "final_train_loss": null,
  "final_val_loss": null,
  "dataset": "WikiText-103",
  "sequence_length": 128,
  "batch_size": null,
  "learning_rate_schedule": null,
  "precision": null
}

Saved baseline_50k.json


In [19]:
# Inspect common model and training configuration attributes
for name in [
    "batch_size", "BATCH_SIZE",
    "block_size", "context_length", "seq_len",
    "vocab_size", "learning_rate", "lr",
    "dropout", "weight_decay", "max_iters",
    "train_losses", "val_losses"
]:
    if name in globals():
        value = globals()[name]
        if name in ("train_losses", "val_losses"):
            value = value[-1] if len(value) else "empty"
        print(f"{name}: {value}")

print("\nModel configuration:")
for attr in ["config", "n_layer", "n_head", "n_embd",
             "num_layers", "num_heads", "dim", "vocab_size"]:
    if hasattr(model, attr):
        print(f"model.{attr}: {getattr(model, attr)}")

batch_size: 16
vocab_size: 32000
learning_rate: 0.0003
weight_decay: 0.01
max_iters: 50000

Model configuration:
model.num_layers: 4
model.num_heads: 4
model.vocab_size: 32000


In [20]:
import torch

# Check common training variables
names = [
    "tokenizer", "dropout", "dropout_rate",
    "scheduler", "lr_scheduler", "precision",
    "dtype", "train_dataset", "val_dataset",
    "train_data", "val_data", "block_size",
    "context_length", "n_embd", "dim",
    "hidden_dim", "ffn_dim", "intermediate_size"
]

print("=== Training configuration ===")
for name in names:
    if name in globals():
        obj = globals()[name]
        if name == "tokenizer":
            print(f"{name}: {type(obj).__name__}")
            for attr in ["vocab_size", "name_or_path"]:
                if hasattr(obj, attr):
                    print(f"  {attr}: {getattr(obj, attr)}")
        elif name in ["train_dataset", "val_dataset",
                      "train_data", "val_data"]:
            print(f"{name}: type={type(obj).__name__}, "
                  f"length={len(obj) if hasattr(obj, '__len__') else 'unknown'}")
        elif name in ["scheduler", "lr_scheduler"]:
            print(f"{name}: {type(obj).__name__}")
        else:
            print(f"{name}: {obj}")

print("\n=== Model attributes ===")
for attr in [
    "dim", "n_embd", "hidden_dim", "ffn_dim",
    "intermediate_size", "dropout", "config"
]:
    if hasattr(model, attr):
        print(f"model.{attr}: {getattr(model, attr)}")

print("\n=== Parameter data types ===")
print({str(p.dtype) for p in model.parameters()})

print("\n=== Attention / FFN modules ===")
for name, module in model.named_modules():
    if any(k in name.lower() for k in ["attn", "attention", "ffn", "mlp", "feed"]):
        if len(list(module.children())) == 0:
            print(name, type(module).__name__)

=== Training configuration ===
tokenizer: Tokenizer
train_data: type=Tensor, length=117273053
val_data: type=Tensor, length=245674

=== Model attributes ===

=== Parameter data types ===
{'torch.float32'}

=== Attention / FFN modules ===
_orig_mod.blocks.0.attention_norm LayerNorm
_orig_mod.blocks.0.attention.qkv Linear
_orig_mod.blocks.0.mlp_norm LayerNorm
_orig_mod.blocks.0.mlp.0 Linear
_orig_mod.blocks.0.mlp.1 ReLU
_orig_mod.blocks.0.mlp.2 Linear
_orig_mod.blocks.1.attention_norm LayerNorm
_orig_mod.blocks.1.attention.qkv Linear
_orig_mod.blocks.1.mlp_norm LayerNorm
_orig_mod.blocks.1.mlp.0 Linear
_orig_mod.blocks.1.mlp.1 ReLU
_orig_mod.blocks.1.mlp.2 Linear
_orig_mod.blocks.2.attention_norm LayerNorm
_orig_mod.blocks.2.attention.qkv Linear
_orig_mod.blocks.2.mlp_norm LayerNorm
_orig_mod.blocks.2.mlp.0 Linear
_orig_mod.blocks.2.mlp.1 ReLU
_orig_mod.blocks.2.mlp.2 Linear
_orig_mod.blocks.3.attention_norm LayerNorm
_orig_mod.blocks.3.attention.qkv Linear
_orig_mod.blocks.3.mlp_norm La

In [21]:
# Inspect tokenizer details
print("=== Tokenizer ===")
print("Type:", type(tokenizer))
print("Attributes:", [
    a for a in dir(tokenizer)
    if any(k in a.lower() for k in
           ["vocab", "encode", "decode", "name", "model", "special"])
])

# Inspect model architecture, including FFN dimensions
print("\n=== Linear layer dimensions ===")
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        print(f"{name}: {module.in_features} -> {module.out_features}")

# Inspect dropout modules and their probabilities
print("\n=== Dropout ===")
found_dropout = False
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Dropout):
        print(name, "p =", module.p)
        found_dropout = True
if not found_dropout:
    print("No nn.Dropout modules found; check for functional dropout.")

# Inspect scheduler variables
print("\n=== Scheduler candidates ===")
for name in ["scheduler", "lr_scheduler", "scaler"]:
    if name in globals():
        obj = globals()[name]
        print(name, "=", type(obj).__name__)

# Inspect dataset tensor properties
print("\n=== Data tensors ===")
print("Train:", train_data.shape, train_data.dtype)
print("Validation:", val_data.shape, val_data.dtype)

=== Tokenizer ===
Type: <class 'tokenizers.Tokenizer'>
Attributes: ['add_special_tokens', 'async_decode_batch', 'async_encode', 'async_encode_batch', 'async_encode_batch_fast', 'decode', 'decode_batch', 'decoder', 'encode', 'encode_batch', 'encode_batch_fast', 'encode_special_tokens', 'get_added_tokens_decoder', 'get_vocab', 'get_vocab_size', 'model', 'num_special_tokens_to_add']

=== Linear layer dimensions ===
_orig_mod.blocks.0.attention.qkv: 256 -> 768
_orig_mod.blocks.0.mlp.0: 256 -> 512
_orig_mod.blocks.0.mlp.2: 512 -> 256
_orig_mod.blocks.1.attention.qkv: 256 -> 768
_orig_mod.blocks.1.mlp.0: 256 -> 512
_orig_mod.blocks.1.mlp.2: 512 -> 256
_orig_mod.blocks.2.attention.qkv: 256 -> 768
_orig_mod.blocks.2.mlp.0: 256 -> 512
_orig_mod.blocks.2.mlp.2: 512 -> 256
_orig_mod.blocks.3.attention.qkv: 256 -> 768
_orig_mod.blocks.3.mlp.0: 256 -> 512
_orig_mod.blocks.3.mlp.2: 512 -> 256

=== Dropout ===
No nn.Dropout modules found; check for functional dropout.

=== Scheduler candidates ===

=

In [22]:
print("Tokenizer model:", tokenizer.model)
print("Tokenizer vocabulary:", tokenizer.get_vocab_size())
print("Special tokens:", tokenizer.get_added_tokens_decoder())

Tokenizer model: BPE(dropout=None, unk_token="<|unk|>", continuing_subword_prefix=None, end_of_word_suffix=None, fuse_unk=False, byte_fallback=False, ignore_merges=False, vocab={"<|unk|>":0, "!":1, """:2, "#":3, "$":4, ...}, merges=[("Ġ", "t"), ("h", "e"), ("Ġ", "a"), ("i", "n"), ("Ġt", "he"), ...])
Tokenizer vocabulary: 32000
Special tokens: {0: AddedToken("<|unk|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True)}
